# ReAct travel agent

This notebook builds a travel-planning agent that uses tools for city facts, currency conversion, route search, itineraries, and climate. It starts from a handwritten OpenAI-style tool JSON schema and a manual ReAct loop, then shows several LangChain ways to declare the same tools (`@tool`, Pydantic `args_schema`, `StructuredTool.from_function`, a stateful `BaseTool` subclass). It also covers `ToolException` handling and a run on 8 test questions, including a missing city (Atlantis) and a request the agent should refuse (booking a hotel).

The dataset is local and synthetic: 8 cities, 19 routes, and FX rates in `travel_data.json`.

**How to run.** Install the packages from `requirements.txt`. By default the notebook talks to a local [Ollama](https://ollama.com) model at `http://localhost:11434/v1` and needs no API key. The outputs already saved in this notebook were produced with `nvidia/nemotron-3-ultra-550b-a55b:free` via OpenRouter; to reproduce them, set `LLM_BASE_URL`, `LLM_MODEL`, and `LLM_API_KEY` as environment variables (do not paste a key into the notebook). See the README for details.


## 0. Setup


In [1]:
%pip install -q "langchain>=1.4,<2" "langchain-openai>=1.7,<2"


In [ ]:
from pathlib import Path

REQUIRED_FILES = ["travel_data.json", "scripted_llm.py"]
missing = [name for name in REQUIRED_FILES if not Path(name).exists()]
if missing:
    raise FileNotFoundError("Put these files next to the notebook: " + ", ".join(missing))


In [3]:
import json
import os
from typing import Annotated, Any, Literal, Optional

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import BaseTool, ToolException, tool
from langchain_core.tools import StructuredTool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_openai import ChatOpenAI
from langgraph.errors import GraphRecursionError
from pydantic import BaseModel, Field, PrivateAttr

from scripted_llm import ScriptedChatModel, tool_call

# Default: local Ollama (OpenAI-compatible API; the "ollama" key is a placeholder).
# The saved outputs in this notebook were produced with
#   LLM_BASE_URL=https://openrouter.ai/api/v1
#   LLM_MODEL=nvidia/nemotron-3-ultra-550b-a55b:free
# To reproduce that run, export those variables plus LLM_API_KEY. Do not hard-code a key.
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "qwen3:8b")
os.environ.setdefault("LLM_API_KEY", "ollama")

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=os.environ["LLM_API_KEY"],
    temperature=0,
)

print("model:", LLM_MODEL)
print("base_url:", LLM_BASE_URL)
print(llm.invoke("Ответь одним словом: готово").content)


Готово


### Данные

Ниже — готовые вспомогательные функции. Их не нужно менять, используйте их в своих инструментах:

* `TRAVEL_DATA` — весь датасет; `CITIES`, `ROUTES`, `RATES` — его части;
* `normalize_city(name)` — приводит «питер», «Istanbul», «ЕРЕВАН» к ключу из `CITIES` или возвращает `None`;
* `MONTHS_IN` — названия месяцев в предложном падеже («в январе»).


In [4]:
TRAVEL_DATA = json.loads(Path("travel_data.json").read_text(encoding="utf-8"))
CITIES: dict = TRAVEL_DATA["cities"]
ROUTES: list = TRAVEL_DATA["routes"]
RATES: dict = TRAVEL_DATA["rates_per_eur"]

TRANSPORT_RU = {"plane": "самолёт", "train": "поезд", "bus": "автобус"}
MONTHS_IN = [
    "январе", "феврале", "марте", "апреле", "мае", "июне",
    "июле", "августе", "сентябре", "октябре", "ноябре", "декабре",
]

def normalize_city(name: str) -> Optional[str]:
    """Возвращает название города, как оно записано в CITIES, или None."""
    key = name.strip().lower().replace("ё", "е")
    for city, info in CITIES.items():
        variants = [city, *info["aliases"]]
        if any(key == variant.lower().replace("ё", "е") for variant in variants):
            return city
    return None

print(TRAVEL_DATA["_about"])
print("Города:", ", ".join(CITIES))
print("Валюты:", RATES)
print("Пример маршрута:", ROUTES[0])
print("Пример города:", json.dumps(CITIES["Тбилиси"], ensure_ascii=False)[:300], "...")


Synthetic travel dataset: cities, routes, climate, and FX rates. Routes are bidirectional; route prices are in EUR. Rates are units of currency per 1 EUR. All figures are invented.
Города: Москва, Санкт-Петербург, Казань, Стамбул, Тбилиси, Ереван, Белград, Дубай
Валюты: {'EUR': 1, 'USD': 1.08, 'RUB': 100, 'TRY': 45, 'GEL': 3, 'AMD': 420, 'RSD': 117, 'AED': 4}
Пример маршрута: {'origin': 'Москва', 'destination': 'Санкт-Петербург', 'transport': 'train', 'duration_hours': 4, 'price_eur': 40}
Пример города: {"aliases": ["Tbilisi"], "country": "Грузия", "currency": "GEL", "utc_offset": 4, "hotel_per_night": 150, "description": "Старый город, серные бани и грузинская кухня.", "attractions": [{"name": "Крепость Нарикала", "price": 0}, {"name": "Серные бани", "price": 80}, {"name": "Канатная дорога", "pric ...


## 1. Агентный цикл с нуля

Прежде чем пользоваться готовым `create_agent`, соберём агента руками, чтобы увидеть протокол tool calling целиком:

1. Модели передают список инструментов (их JSON-схемы) — `llm.bind_tools(tools)`.
2. Модель отвечает `AIMessage`. Если ей нужен инструмент, в `response.tool_calls` будет список вида `{"name": ..., "args": {...}, "id": ...}`. Сама модель ничего не выполняет.
3. Мы вызываем функцию и кладём результат в историю как `ToolMessage(content=..., tool_call_id=<тот же id>)`.
4. Снова вызываем модель с обновлённой историей. Повторяем, пока модель не ответит текстом без `tool_calls`.

Это и есть ReAct-цикл: рассуждение → действие → наблюдение → …

Чтобы агенту было что вызывать, нужен хотя бы один инструмент. В этой части он будет один — пересчёт валют `convert_currency`. Это обычная Python-функция: про агентские фреймворки (к примеру, LangChain) она ничего не знает. Дальше вы опишете её для модели (1.1) и напишете цикл, в котором модель сможет её вызывать (1.2).


In [5]:
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Конвертирует сумму между валютами по курсам из учебного датасета."""
    source, target = from_currency.strip().upper(), to_currency.strip().upper()
    unknown = [code for code in (source, target) if code not in RATES]
    if unknown:
        raise ToolException(f"Неизвестная валюта: {', '.join(unknown)}. Доступны: {', '.join(RATES)}")
    result = amount / RATES[source] * RATES[target]
    return f"{amount:.2f} {source} = {result:.2f} {target}"

convert_currency(100, "EUR", "RUB")


'100.00 EUR = 10000.00 RUB'

### 1.1. JSON-схема инструмента

Первый и самый низкоуровневый способ передать инструмент модели — описать его JSON-схемой в формате OpenAI function calling. Модель не видит Python-код: она получает только имя инструмента, описание и схему аргументов. Какой бы способ задать инструмент вы ни выбрали дальше (декоратор, Pydantic, класс), LangChain в итоге превратит его именно в такую схему.

Опишите `convert_currency` в этом формате вручную, словарём:

```python
{"type": "function", "function": {"name": ..., "description": ..., "parameters": <JSON Schema>}}
```

Аргументы описываются внутри `parameters` в формате JSON Schema: `parameters` — это объект (`"type": "object"`), у которого в `properties` перечислены аргументы, а в `required` — обязательные из них. Например, для функции `get_weather(city: str, days: int)` схема выглядит так:

```python
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Возвращает прогноз погоды в городе на несколько дней вперёд.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Название города, например «Казань»"},
                "days": {"type": "integer", "description": "На сколько дней прогноз, от 1 до 7"},
            },
            "required": ["city", "days"],
        },
    },
}
```

Для `convert_currency`:

* в `properties` три аргумента: `amount` (number), `from_currency` и `to_currency` (string), все три обязательные;
* у инструмента и у каждого аргумента должно быть описание. Подумайте, что модели нужно знать, чтобы правильно заполнить `from_currency`: «рубли» или `RUB`?


In [6]:
CONVERT_CURRENCY_SCHEMA = {
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description": (
            "Конвертирует сумму из одной валюты в другую по учебным курсам. "
            "Используй ISO-коды валют (EUR, USD, RUB, TRY, GEL, AMD, RSD, AED)."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "Сумма для конвертации",
                },
                "from_currency": {
                    "type": "string",
                    "description": "Исходная валюта в формате ISO-кода, например EUR или RUB (не «рубли»)",
                },
                "to_currency": {
                    "type": "string",
                    "description": "Целевая валюта в формате ISO-кода, например RUB или GEL",
                },
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
}



### 1.2. ReAct-цикл

Напишите `run_manual_agent(llm, question, tools, registry, max_steps=5, max_tool_errors=2)`.

* **Вход:** модель, вопрос пользователя, список схем инструментов (`tools`), словарь `registry` «имя инструмента → Python-функция», лимит обращений к модели и лимит ошибок инструментов подряд.
* **Выход:** `dict` с ключами `answer` (текст финального ответа), `messages` (вся история, включая `ToolMessage`), `steps` (сколько раз вызвали модель).
* История начинается с `SystemMessage(MANUAL_SYSTEM_PROMPT)` и `HumanMessage(question)`.
* На каждый вызов из `tool_calls` в историю добавляется ровно один `ToolMessage` с тем же `tool_call_id`.

**Ошибки.** Модель может ошибиться в аргументах (например, передать «рубли» вместо `RUB`) и исправиться, если увидит текст ошибки. Но она может и повторять один и тот же неудачный вызов, сжигая шаги. Поэтому:

* если инструмент бросил исключение, текст ошибки уходит модели в `ToolMessage`;
* если модель вызвала несуществующий инструмент, в `ToolMessage` уходит сообщение «такого инструмента нет» со списком доступных — это тоже ошибка;
* считайте ошибки подряд: каждый неудачный вызов увеличивает счётчик, успешный — обнуляет;
* когда после обработки ответа модели счётчик достиг `max_tool_errors`, цикл останавливается, а в `answer` — понятное пользователю сообщение с текстом последней ошибки.

**Лимит шагов.** Если за `max_steps` обращений модель так и не ответила текстом, верните в `answer` понятное сообщение (не `None`).

Для отладки без трат на API в `scripted_llm.py` есть `ScriptedChatModel` — фейковая модель, которая отвечает заранее заданными сообщениями (пример в docstring).


<details>
<summary><strong>Подсказка</strong></summary>

`for step in range(1, max_steps + 1): response = model.invoke(messages) ...` — и не забудьте добавить сам `response` в историю перед `ToolMessage`: модель должна видеть, какой вызов она сделала. Если в одном ответе несколько `tool_calls`, обработайте их все и только потом проверяйте счётчик ошибок.

</details>


In [7]:
MANUAL_SYSTEM_PROMPT = (
    "Ты помощник по путешествиям. Для пересчёта валют обязательно используй инструмент, "
    "не считай курсы сам. Отвечай кратко, по-русски."
)

def run_manual_agent(
    llm, question: str, tools: list, registry: dict, max_steps: int = 5, max_tool_errors: int = 2
) -> dict:
    """Минимальный ReAct-агент без фреймворка агентов.

    Args:
        llm: Chat-модель LangChain с поддержкой bind_tools.
        question: Вопрос пользователя.
        tools: Схемы инструментов для модели.
        registry: Имя инструмента → Python-функция.
        max_steps: Максимальное число обращений к модели.
        max_tool_errors: После скольких ошибок инструментов подряд остановиться.

    Returns:
        {"answer": str, "messages": list, "steps": int}
    """
    # привязываем схемы инструментов к модели
    model = llm.bind_tools(tools)
    messages = [SystemMessage(MANUAL_SYSTEM_PROMPT), HumanMessage(question)]
    consecutive_errors = 0
    last_error_text = ""

    for step in range(1, max_steps + 1):
        response = model.invoke(messages)
        messages.append(response)  # модель должна видеть свой tool_call

        # финальный текстовый ответ без вызовов
        if not response.tool_calls:
            return {
                "answer": response.content or "",
                "messages": messages,
                "steps": step,
            }

        # обрабатываем ВСЕ tool_calls из одного ответа, потом проверяем счётчик
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]
            call_id = call["id"]

            if name not in registry:
                available = ", ".join(registry)
                err = f"Неизвестный инструмент: {name}. Доступны: {available}"
                messages.append(ToolMessage(content=err, tool_call_id=call_id))
                consecutive_errors += 1
                last_error_text = err
                continue

            try:
                result = registry[name](**args)
                messages.append(ToolMessage(content=str(result), tool_call_id=call_id))
                consecutive_errors = 0  # успешный вызов обнуляет счётчик
            except Exception as error:  # ToolException и прочие — текст модели
                err = str(error)
                messages.append(ToolMessage(content=err, tool_call_id=call_id))
                consecutive_errors += 1
                last_error_text = err

        if consecutive_errors >= max_tool_errors:
            return {
                "answer": (
                    f"Останавливаюсь: слишком много ошибок инструментов подряд. "
                    f"Последняя ошибка: {last_error_text}"
                ),
                "messages": messages,
                "steps": step,
            }

    return {
        "answer": f"Превышен лимит шагов ({max_steps}). Не удалось получить финальный ответ.",
        "messages": messages,
        "steps": max_steps,
    }



Запустим на настоящей модели. Вопрос требует двух вызовов инструмента: посмотрите, делает ли модель их параллельно (два `tool_calls` в одном ответе) или по очереди.


In [8]:
result = run_manual_agent(
    llm,
    "Сколько будет 250 евро в грузинских лари и в армянских драмах?",
    tools=[CONVERT_CURRENCY_SCHEMA],
    registry={"convert_currency": convert_currency},
)
for message in result["messages"]:
    message.pretty_print()
print("Обращений к модели:", result["steps"])


================================ System Message ================================

Ты помощник по путешествиям. Для пересчёта валют обязательно используй инструмент, не считай курсы сам. Отвечай кратко, по-русски.
================================ Human Message =================================

Сколько будет 250 евро в грузинских лари и в армянских драмах?
================================== Ai Message ==================================
Tool Calls:
  convert_currency (call-1bb10711-13fb-4aff-b2e9-4a25025586cc)
 Call ID: call-1bb10711-13fb-4aff-b2e9-4a25025586cc
  Args:
    amount: 250
    from_currency: EUR
    to_currency: GEL
  convert_currency (call-64c29616-2e2f-4e93-98dd-db8bbd45de44)
 Call ID: call-64c29616-2e2f-4e93-98dd-db8bbd45de44
  Args:
    to_currency: AMD
    amount: 250
    from_currency: EUR
================================= Tool Message =================================

250.00 EUR = 750.00 GEL
================================= Tool Message ==============================

## 2. Пять способов задать инструмент

Писать JSON-схему руками утомительно и легко ошибиться. LangChain умеет строить её сам. Попробуем все основные способы на инструментах планировщика:

| № | Способ | Инструмент |
|---|---|---|
| 2.1 | `@tool` + аннотации типов + docstring | `get_city_info` |
| 2.2 | `@tool(..., args_schema=PydanticModel)` | `search_routes` |
| 2.3 | `StructuredTool.from_function(...)` | `convert_currency_tool` |
| 2.4 | Класс-наследник `BaseTool` с состоянием | `ItineraryTool` |
| 2.5 | Починка плохо описанного инструмента | `get_climate` |

Для каждого инструмента полезно смотреть, что именно увидит модель: `convert_to_openai_tool(tool)` возвращает ту же структуру, что вы писали руками в 1.1.

Во всех инструментах неизвестный город — это ошибка: бросайте `ToolException` с понятным текстом (что не так и какие города есть). Что с ней делать дальше, разберёмся в части 4.


### 2.1. Декоратор `@tool`

Напишите `get_city_info(city: str) -> str`: справку о городе — страну, валюту, часовой пояс, цену отеля за ночь, описание и достопримечательности с ценами (0 — «бесплатно»).

* Имя и описание инструмента берутся из имени функции и docstring.
* Чтобы у аргумента тоже появилось описание, используйте `@tool(parse_docstring=True)` и раздел `Args:` в docstring (Google style).
* Город ищите через `normalize_city`.


In [9]:
@tool(parse_docstring=True)
def get_city_info(city: str) -> str:
    """Возвращает справку о городе: страна, валюта, часовой пояс, цена отеля и достопримечательности.

    Args:
        city: Название города или его алиас, например «Ереван» или «Питер».
    """
    name = normalize_city(city)
    if name is None:
        raise ToolException(f"Неизвестный город: {city}. Доступны: {', '.join(CITIES)}")
    info = CITIES[name]
    # «Каскад (бесплатно); Матенадаран (2500 AMD); ...»
    parts = []
    for attraction in info["attractions"]:
        if attraction["price"] == 0:
            parts.append(f"{attraction['name']} (бесплатно)")
        else:
            parts.append(f"{attraction['name']} ({attraction['price']} {info['currency']})")
    sights = "; ".join(parts)
    return (
        f"{name}, {info['country']}. Валюта: {info['currency']}. Часовой пояс: UTC+{info['utc_offset']}.\n"
        f"Отель: около {info['hotel_per_night']} {info['currency']} за ночь.\n"
        f"{info['description']}\n"
        f"Достопримечательности: {sights}"
    )

print(get_city_info.invoke({"city": "ереван"}))
print(convert_to_openai_tool(get_city_info))


Ереван, Армения. Валюта: AMD. Часовой пояс: UTC+4.
Отель: около 25000 AMD за ночь.
Розовый город с видом на Арарат.
Достопримечательности: Каскад (бесплатно); Матенадаран (2500 AMD); Храм Гарни (1500 AMD)
{'type': 'function', 'function': {'name': 'get_city_info', 'description': 'Возвращает справку о городе: страна, валюта, часовой пояс, цена отеля и достопримечательности.', 'parameters': {'properties': {'city': {'description': 'Название города или его алиас, например «Ереван» или «Питер».', 'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


### 2.2. Явная схема через Pydantic

Когда у аргументов есть ограничения (допустимые значения, диапазоны, значения по умолчанию), удобнее описать их Pydantic-моделью и передать через `args_schema`.

**Pydantic-модель данных** — это класс-наследник `pydantic.BaseModel`, поля которого описывают данные: тип, значение по умолчанию, ограничения и описание (`Field(...)`). LangChain строит из неё JSON-схему аргументов для модели, а перед вызовом инструмента проверяет, что аргументы, которые сгенерировала модель, ей соответствуют. Если нет — инструмент не вызывается, а модель получает текст ошибки валидации.

Напишите модель данных `RouteQuery` и инструмент `search_routes`:

* `origin`, `destination` — обязательные строки;
* `transport` — `Literal["plane", "train", "bus", "any"]`, по умолчанию `"any"`;
* `max_price_eur` — необязательное число, `≥ 0` (`Field(ge=0)`), по умолчанию `None`;
* у всех полей есть `description`.

Инструмент ищет прямые маршруты, учитывая, что маршруты двусторонние (Москва → Казань = Казань → Москва), фильтрует по транспорту и цене и возвращает их отсортированными по цене, по строке на маршрут, например `Москва → Стамбул: самолёт, 4 ч, 250 EUR`. Если ничего не нашлось — понятный текст «маршрутов не найдено».

Подумайте, что написать в описании инструмента, чтобы агент догадался искать пересадку, когда прямого маршрута нет.


In [10]:
class RouteQuery(BaseModel):
    """Параметры поиска прямых маршрутов между двумя городами."""

    origin: str = Field(description="Город отправления, например «Москва» или «Питер»")
    destination: str = Field(description="Город назначения")
    transport: Literal["plane", "train", "bus", "any"] = Field(
        default="any",
        description="Тип транспорта: plane, train, bus или any (любой)",
    )
    max_price_eur: Optional[float] = Field(
        default=None,
        ge=0,
        description="Максимальная цена в евро; если не указана — без ограничения по цене",
    )

@tool(args_schema=RouteQuery)
def search_routes(
    origin: str, destination: str, transport: str = "any", max_price_eur: Optional[float] = None
) -> str:
    """Ищет прямые маршруты между двумя городами (маршруты двусторонние).

    Возвращает варианты, отсортированные по цене (дешёвые первыми).
    Если прямого маршрута нет — так и скажи; для дальней дороги можно искать
    пересадку через другие города отдельными вызовами этого же инструмента.
    """
    origin_name = normalize_city(origin)
    dest_name = normalize_city(destination)
    if origin_name is None:
        raise ToolException(
            f"Неизвестный город отправления: {origin}. Доступны: {', '.join(CITIES)}"
        )
    if dest_name is None:
        raise ToolException(
            f"Неизвестный город назначения: {destination}. Доступны: {', '.join(CITIES)}"
        )

    found = []
    for route in ROUTES:
        ends = {route["origin"], route["destination"]}
        if ends != {origin_name, dest_name}:
            continue
        if transport != "any" and route["transport"] != transport:
            continue
        if max_price_eur is not None and route["price_eur"] > max_price_eur:
            continue
        found.append(route)

    found.sort(key=lambda item: item["price_eur"])
    if not found:
        return f"Прямых маршрутов из {origin_name} в {dest_name} не найдено."

    lines = []
    for route in found:
        transport_ru = TRANSPORT_RU.get(route["transport"], route["transport"])
        lines.append(
            f"{origin_name} → {dest_name}: {transport_ru}, "
            f"{route['duration_hours']} ч, {route['price_eur']} EUR"
        )
    return "\n".join(lines)

print(search_routes.invoke({"origin": "Питер", "destination": "Москва"}))
print(convert_to_openai_tool(search_routes))


Санкт-Петербург → Москва: автобус, 10 ч, 20 EUR
Санкт-Петербург → Москва: поезд, 4 ч, 40 EUR
Санкт-Петербург → Москва: самолёт, 1.5 ч, 60 EUR
{'type': 'function', 'function': {'name': 'search_routes', 'description': 'Ищет прямые маршруты между двумя городами (маршруты двусторонние).\n\nВозвращает варианты, отсортированные по цене (дешёвые первыми).\nЕсли прямого маршрута нет — так и скажи; для дальней дороги можно искать\nпересадку через другие города отдельными вызовами этого же инструмента.', 'parameters': {'properties': {'origin': {'description': 'Город отправления, например «Москва» или «Питер»', 'type': 'string'}, 'destination': {'description': 'Город назначения', 'type': 'string'}, 'transport': {'default': 'any', 'description': 'Тип транспорта: plane, train, bus или any (любой)', 'enum': ['plane', 'train', 'bus', 'any'], 'type': 'string'}, 'max_price_eur': {'anyOf': [{'minimum': 0, 'type': 'number'}, {'type': 'null'}], 'default': None, 'description': 'Максимальная цена в евро; ес

### 2.3. `StructuredTool.from_function`

Бывает, что функция уже есть (из чужого модуля, из части 1) и менять её код нельзя. Тогда инструмент собирают снаружи. Оберните `convert_currency` из части 1 в `StructuredTool.from_function`, задав имя `convert_currency` и содержательное `description`.

Сравните получившуюся схему со своей схемой из 1.1: чего в ней не хватает и почему? Ответ запишите в 2.6.


In [11]:
# Оборачиваем уже существующую convert_currency, не меняя её код
convert_currency_tool = StructuredTool.from_function(
    func=convert_currency,
    name="convert_currency",
    description=(
        "Конвертирует сумму между валютами по учебным курсам датасета. "
        "Аргументы from_currency и to_currency — ISO-коды (EUR, RUB, GEL, AMD и т.д.)."
    ),
)

print(convert_to_openai_tool(convert_currency_tool))


{'type': 'function', 'function': {'name': 'convert_currency', 'description': 'Конвертирует сумму между валютами по учебным курсам датасета. Аргументы from_currency и to_currency — ISO-коды (EUR, RUB, GEL, AMD и т.д.).', 'parameters': {'properties': {'amount': {'type': 'number'}, 'from_currency': {'type': 'string'}, 'to_currency': {'type': 'string'}}, 'required': ['amount', 'from_currency', 'to_currency'], 'type': 'object'}}}


### 2.4. Наследник `BaseTool` с состоянием

Класс нужен, когда инструменту требуется состояние или конфигурация — подключение к БД, клиент API, накопленные данные. Наш случай: маршрут поездки, который агент собирает за несколько вызовов.

Напишите `ItineraryTool(BaseTool)` с именем `manage_itinerary` и схемой аргументов `ItineraryInput`:

* `action`: `Literal["add", "show", "clear"]`;
* `city`: необязательная строка (нужна для `add`);
* `nights`: необязательное целое от 1 до 30 (нужно для `add`).

Поведение:

* конфигурация — поле `data: dict` (весь `TRAVEL_DATA`), передаётся в конструктор: `ItineraryTool(data=TRAVEL_DATA)`;
* состояние — список остановок в `PrivateAttr`: у каждого экземпляра свой маршрут;
* `add` добавляет город и число ночей; неизвестный город или отсутствие `nights` → `ToolException`;
* `clear` очищает маршрут;
* `show` возвращает маршрут в порядке добавления и итог. Стоимость отеля: `ночи × hotel_per_night / курс валюты города` (курсы — единиц за 1 EUR). Последние две строки строго в формате:

```
Маршрут поездки:
1. Тбилиси — 3 ноч., отель ≈ 150.00 EUR
2. Ереван — 2 ноч., отель ≈ 119.05 EUR
Итого ночей: 5
Стоимость отелей: ≈ 269.05 EUR
```

Для пустого маршрута `show` выводит `Маршрут пуст.` и те же две итоговые строки: `Итого ночей: 0` и `Стоимость отелей: ≈ 0.00 EUR`.


<details>
<summary><strong>Подсказка</strong></summary>

В `BaseTool` поля объявляются как в Pydantic-модели: `name: str = "manage_itinerary"`, `description: str = ...`, `args_schema: type[BaseModel] = ItineraryInput`. Логика — в методе `_run(self, action, city=None, nights=None) -> str`. Состояние храните в `PrivateAttr(default_factory=list)`: обычное поле стало бы частью конфигурации инструмента (его можно передать в конструктор, оно попадает в `repr` и сериализацию), а это внутренние данные.

</details>


In [12]:
class ItineraryInput(BaseModel):
    action: Literal["add", "show", "clear"] = Field(
        description="Действие: add — добавить город, show — показать маршрут, clear — очистить"
    )
    city: Optional[str] = Field(default=None, description="Город для action=add")
    nights: Optional[int] = Field(
        default=None, ge=1, le=30, description="Число ночей (от 1 до 30) для action=add"
    )

class ItineraryTool(BaseTool):
    name: str = "manage_itinerary"
    description: str = (
        "Ведёт маршрут поездки: добавляет города с числом ночей, показывает итог "
        "стоимости отелей в EUR или очищает маршрут."
    )
    args_schema: type[BaseModel] = ItineraryInput
    data: dict  # TRAVEL_DATA, передаётся в конструктор
    _stops: list = PrivateAttr(default_factory=list)  # своё состояние у каждого экземпляра

    def _run(
        self, action: str, city: Optional[str] = None, nights: Optional[int] = None, run_manager=None
    ) -> str:
        if action == "clear":
            self._stops = []
            return "Маршрут очищен."

        if action == "show":
            return self._format_show()

        # action == "add"
        if city is None:
            raise ToolException("Для action=add нужно указать city.")
        if nights is None:
            raise ToolException("Для action=add нужно указать nights (1–30).")
        name = normalize_city(city)
        if name is None:
            raise ToolException(
                f"Неизвестный город: {city}. Доступны: {', '.join(self.data['cities'])}"
            )
        self._stops.append({"city": name, "nights": nights})
        return f"Добавлено: {name}, {nights} ноч."

    def _format_show(self) -> str:
        rates = self.data["rates_per_eur"]
        cities = self.data["cities"]
        if not self._stops:
            return "Маршрут пуст.\nИтого ночей: 0\nСтоимость отелей: ≈ 0.00 EUR"

        lines = ["Маршрут поездки:"]
        total_nights = 0
        total_eur = 0.0
        for index, stop in enumerate(self._stops, start=1):
            name = stop["city"]
            n = stop["nights"]
            info = cities[name]
            # ночи × hotel_per_night / курс (единиц валюты за 1 EUR)
            cost_eur = n * info["hotel_per_night"] / rates[info["currency"]]
            total_nights += n
            total_eur += cost_eur
            lines.append(f"{index}. {name} — {n} ноч., отель ≈ {cost_eur:.2f} EUR")
        lines.append(f"Итого ночей: {total_nights}")
        lines.append(f"Стоимость отелей: ≈ {total_eur:.2f} EUR")
        return "\n".join(lines)

itinerary_tool = ItineraryTool(data=TRAVEL_DATA)
print(convert_to_openai_tool(itinerary_tool))


{'type': 'function', 'function': {'name': 'manage_itinerary', 'description': 'Ведёт маршрут поездки: добавляет города с числом ночей, показывает итог стоимости отелей в EUR или очищает маршрут.', 'parameters': {'properties': {'action': {'description': 'Действие: add — добавить город, show — показать маршрут, clear — очистить', 'enum': ['add', 'show', 'clear'], 'type': 'string'}, 'city': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Город для action=add'}, 'nights': {'anyOf': [{'maximum': 30, 'minimum': 1, 'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Число ночей (от 1 до 30) для action=add'}}, 'required': ['action'], 'type': 'object'}}}


### 2.5. Починить плохой инструмент

Вот инструмент, который «работает». Посмотрите на его схему: что из неё может понять модель?


In [13]:
@tool
def climate(c, m):
    """climate"""
    return CITIES[c]["climate"][m - 1]

print(climate.invoke({"c": "Тбилиси", "m": 5}))
print(convert_to_openai_tool(climate))


17
{'type': 'function', 'function': {'name': 'climate', 'description': 'climate', 'parameters': {'properties': {'c': {}, 'm': {}}, 'required': ['c', 'm'], 'type': 'object'}}}


Перепишите его как `get_climate(city, month) -> str`:

* понятное имя, типы аргументов, описание инструмента и каждого аргумента;
* `month` — целое от 1 до 12 прямо в схеме (подсказка: `Annotated[int, Field(ge=1, le=12)]`);
* город — через `normalize_city`, неизвестный город → `ToolException`;
* ответ — фраза вида «Средняя температура в городе Дубай в июле: +35 °C» (`MONTHS_IN` уже есть).

Здесь можно использовать любой из способов выше — выберите тот, что кажется уместнее, и объясните выбор в 2.6.


In [14]:
@tool(parse_docstring=True)
def get_climate(
    city: str,
    month: Annotated[int, Field(ge=1, le=12)],
) -> str:
    """Возвращает среднюю температуру в городе за указанный месяц.

    Args:
        city: Название города или алиас, например «Дубай».
        month: Номер месяца от 1 (январь) до 12 (декабрь).
    """
    name = normalize_city(city)
    if name is None:
        raise ToolException(f"Неизвестный город: {city}. Доступны: {', '.join(CITIES)}")
    temp = CITIES[name]["climate"][month - 1]
    sign = "+" if temp > 0 else ""
    return f"Средняя температура в городе {name} в {MONTHS_IN[month - 1]}: {sign}{temp} °C"

print(get_climate.invoke({"city": "Дубай", "month": 7}))
print(convert_to_openai_tool(get_climate))


Средняя температура в городе Дубай в июле: +35 °C
{'type': 'function', 'function': {'name': 'get_climate', 'description': 'Возвращает среднюю температуру в городе за указанный месяц.', 'parameters': {'properties': {'city': {'description': 'Название города или алиас, например «Дубай».', 'type': 'string'}, 'month': {'description': 'Номер месяца от 1 (январь) до 12 (декабрь).', 'maximum': 12, 'minimum': 1, 'type': 'integer'}}, 'required': ['city', 'month'], 'type': 'object'}}}


### Итоговый набор инструментов

Соберём все инструменты в один список и посмотрим их схемы рядом.


In [15]:
ALL_TOOLS = [get_city_info, search_routes, convert_currency_tool, itinerary_tool, get_climate]

for current_tool in ALL_TOOLS:
    print(json.dumps(convert_to_openai_tool(current_tool), ensure_ascii=False, indent=1))
    print()


{
 "type": "function",
 "function": {
  "name": "get_city_info",
  "description": "Возвращает справку о городе: страна, валюта, часовой пояс, цена отеля и достопримечательности.",
  "parameters": {
   "properties": {
    "city": {
     "description": "Название города или его алиас, например «Ереван» или «Питер».",
     "type": "string"
    }
   },
   "required": [
    "city"
   ],
   "type": "object"
  }
 }
}

{
 "type": "function",
 "function": {
  "name": "search_routes",
  "description": "Ищет прямые маршруты между двумя городами (маршруты двусторонние).\n\nВозвращает варианты, отсортированные по цене (дешёвые первыми).\nЕсли прямого маршрута нет — так и скажи; для дальней дороги можно искать\nпересадку через другие города отдельными вызовами этого же инструмента.",
  "parameters": {
   "properties": {
    "origin": {
     "description": "Город отправления, например «Москва» или «Питер»",
     "type": "string"
    },
    "destination": {
     "description": "Город назначения",
     

### 2.6. Сравнение способов

Ответьте своими словами, опираясь на схемы выше (5–10 предложений):

1. Чем отличаются схемы, полученные разными способами? Чего не хватает схеме `convert_currency_tool` по сравнению с вашей ручной из 1.1 и как это исправить, не трогая функцию?
2. Когда вы выберете `@tool`, когда `args_schema`, когда `StructuredTool.from_function`, а когда наследника `BaseTool`? Приведите по примеру из реальной задачи.
3. Что модель «знает» об инструменте, кроме его схемы? Почему описания так важны?

**Analysis / Разбор:**

1. Схемы отличаются тем, откуда берутся описания аргументов. У @tool(parse_docstring=True) они берутся из раздела Args: в docstring. У args_schema=RouteQuery — из Field(description=...), и ещё появляются enum, default и minimum. У наследника BaseTool схема берётся из args_schema, а имя и описание — из полей класса. В convert_currency_tool у аргументов нет описаний, потому что в docstring исходной функции нет раздела Args:. Это можно исправить, не меняя функцию: передать в from_function свою Pydantic-модель через args_schema.
2. @tool подходит для простых функций, где хватает docstring (get_city_info, get_climate). args_schema нужен, когда у аргументов есть ограничения (Literal, ge/le, значения по умолчанию), как в search_routes. StructuredTool.from_function удобен, если функция уже есть и менять её нельзя. Наследник BaseTool нужен, когда инструмент хранит состояние, например маршрут в ItineraryTool.
3. Модель не видит Python-код, только имя, описание и JSON-схему. Поэтому от описаний зависит, вызовет ли она инструмент и с какими аргументами.


## 3. Агент на `create_agent`

Теперь отдадим цикл из части 1 фреймворку. `create_agent(model, tools, system_prompt=...)` из `langchain.agents` собирает ровно тот же ReAct-цикл, но умеет больше: исполняет инструменты параллельно, обрабатывает ошибки валидации аргументов.

**Память диалога.** Агент сам по себе ничего не помнит между вызовами `invoke`: он видит только те сообщения, которые вы ему передали. Поэтому память диалога — это просто список сообщений, как `messages` в части 1. `agent.invoke({"messages": [...]})` возвращает словарь, где `result["messages"]` — вся история: переданные сообщения плюс новые (вызовы инструментов, их результаты, ответ). Чтобы продолжить диалог, передайте эту историю в следующий вызов вместе с новой репликой. Новый диалог — пустой список.

### 3. Системный промпт и агент

1. Напишите `SYSTEM_PROMPT` (не короче 200 символов): роль агента, правила работы с инструментами (факты только из инструментов, что делать, если прямого маршрута нет, в какой валюте цены), формат ответа и что делать с запросами, которые инструментами не решаются.
2. `build_agent(llm, tools)` — возвращает агента из `create_agent` с вашим промптом.
3. `chat(agent, history, message) -> (answer, history)` — добавляет реплику пользователя к истории `history`, вызывает агента и возвращает пару: текст последнего ответа агента и обновлённую историю. Исходный список `history` не изменяйте.


In [16]:
SYSTEM_PROMPT = """
Ты — агент-планировщик поездок. Помогаешь выбрать направление, найти маршруты,
пересчитать цены в местную валюту и вести маршрут поездки.

Правила работы с инструментами:
1. Факты о городах, маршрутах, курсах и климате бери ТОЛЬКО из инструментов, не выдумывай.
2. Цены маршрутов в датасете указаны в EUR; отели — в местной валюте города.
   Для пересчёта валют всегда вызывай convert_currency.
3. Если прямого маршрута нет, ищи пересадку: вызови search_routes через промежуточный
   город (например Стамбул) отдельными вызовами и сложи цены сегментов.
4. get_climate даёт среднюю температуру по месяцам, а не прогноз на конкретный день.
5. Ты НЕ умеешь бронировать отели, покупать билеты и ничего оплачивать.
   Если просят купить/забронировать — честно откажи и предложи, что можешь посчитать.
6. Отвечай кратко, по-русски, с конкретными цифрами из результатов инструментов.
""".strip()

def build_agent(llm, tools: list):
    """Собирает ReAct-агента на create_agent."""
    return create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)

def chat(agent, history: list, message: str) -> tuple[str, list]:
    """Продолжает диалог: возвращает ответ агента и обновлённую историю."""
    # не меняем исходный список history
    new_messages = list(history) + [HumanMessage(message)]
    result = agent.invoke({"messages": new_messages})
    updated = result["messages"]
    answer = ""
    for msg in reversed(updated):
        if isinstance(msg, AIMessage) and not msg.tool_calls:
            answer = msg.content if isinstance(msg.content, str) else str(msg.content)
            break
    return answer, updated



Проведём диалог из трёх реплик. Каждая следующая опирается на предыдущие — агент должен помнить контекст. В конце убедимся, что с пустой историей агент начинает разговор с чистого листа. Перед этим очищаем и маршрут поездки: состояние инструмента — это не память диалога, и без очистки агент «вспомнит» прошлую поездку через `manage_itinerary`.


In [17]:
itinerary_tool.invoke({"action": "clear"})
agent = build_agent(llm, ALL_TOOLS)

dialog = [
    "Выбираю, куда поехать из Москвы в октябре: Стамбул, Тбилиси или Дубай. Где будет теплее +15?",
    "Давай Стамбул. Сколько стоит туда добраться и сколько это в местной валюте?",
    "Добавь в маршрут 4 ночи там и покажи, сколько выйдут отели.",
]
trip_history = []
for message in dialog:
    print("Пользователь:", message)
    answer, trip_history = chat(agent, trip_history, message)
    print("Агент:", answer)
    print()

# Новый диалог: пустая история и пустой маршрут — иначе агент «вспомнит»
# прошлую поездку через manage_itinerary, а не через память диалога
itinerary_tool.invoke({"action": "clear"})
print("Пользователь (новый диалог): Куда я собирался?")
answer, _ = chat(agent, [], "Куда я собирался?")
print("Агент:", answer)


Пользователь: Выбираю, куда поехать из Москвы в октябре: Стамбул, Тбилиси или Дубай. Где будет теплее +15?
Агент: По данным средних температур за октябрь:

- **Стамбул**: +16 °C — **теплее +15**
- **Тбилиси**: +14 °C — чуть прохладнее
- **Дубай**: +29 °C — **очень тепло**

Таким образом, в октябре **Стамбул и Дубай** точно будут теплее +15 °C.

Пользователь: Давай Стамбул. Сколько стоит туда добраться и сколько это в местной валюте?
Агент: **Москва → Стамбул**  
- Прямой рейс: **самолёт, 4 ч, 250 EUR**  
- В местной валюте (турецкая лира, TRY): **≈ 11 250 TRY** (по курсу датасета)

*Отель в Стамбуле стоит около 2 700 TRY за ночь.*

Пользователь: Добавь в маршрут 4 ночи там и покажи, сколько выйдут отели.
Агент: Готово. В маршрут добавлено **4 ночи в Стамбуле**.

**Итог по отелям:**
- Стамбул — 4 ночи ≈ **240 EUR** (по курсу датасета это около 10 800 TRY)

Всего ночей в маршруте: **4**.

Пользователь (новый диалог): Куда я собирался?
Агент: У вас пока пустой маршрут — ни один город не д

Посмотрите, какие инструменты агент вызывал внутри диалога: вся история лежит в `trip_history`.


In [18]:
for message in trip_history:
    message.pretty_print()


================================ Human Message =================================

Выбираю, куда поехать из Москвы в октябре: Стамбул, Тбилиси или Дубай. Где будет теплее +15?
================================== Ai Message ==================================
Tool Calls:
  get_climate (call-68cfc3e9-5efe-4acd-8703-185973d63f7c)
 Call ID: call-68cfc3e9-5efe-4acd-8703-185973d63f7c
  Args:
    city: Стамбул
    month: 10
  get_climate (call-305e8c21-28f9-4a26-a1a4-e527a36933c0)
 Call ID: call-305e8c21-28f9-4a26-a1a4-e527a36933c0
  Args:
    month: 10
    city: Тбилиси
  get_climate (call-f2a97c1d-690a-442c-9ebd-5dbe05e8055b)
 Call ID: call-f2a97c1d-690a-442c-9ebd-5dbe05e8055b
  Args:
    city: Дубай
    month: 10
================================= Tool Message =================================
Name: get_climate

Средняя температура в городе Стамбул в октябре: +16 °C
================================= Tool Message =================================
Name: get_climate

Средняя температура в городе 

## 4. Работа с ошибками

В части 1 вы сами решали, что делать с исключением инструмента. В `create_agent` есть два случая:

* **ошибка валидации аргументов** (например, `max_price_eur=-5`) — агент по умолчанию возвращает модели текст ошибки, и она может исправиться;
* **`ToolException` из кода инструмента** — по умолчанию пробрасывается наружу и роняет весь вызов агента. Проверьте сами ячейкой ниже.


In [19]:
crash_llm = ScriptedChatModel(responses=[tool_call("get_city_info", city="Атлантида"), AIMessage(content="...")])
try:
    build_agent(crash_llm, ALL_TOOLS).invoke({"messages": [{"role": "user", "content": "Расскажи про Атлантиду"}]})
except ToolException as error:
    print("Агент упал:", error)


Агент упал: Неизвестный город: Атлантида. Доступны: Москва, Санкт-Петербург, Казань, Стамбул, Тбилиси, Ереван, Белград, Дубай


### 4.1. Обработка ошибок инструментов

У `BaseTool` есть поле `handle_tool_error`: `True` (вернуть модели текст исключения), строка (вернуть её) или функция `(ToolException) -> str`.

1. Напишите `format_tool_error(error)`: текст для модели, а не для человека — что пошло не так и что ей делать дальше (исправить аргументы, взять другой город, сообщить пользователю). Обязательно включите в него исходный текст исключения (`str(error)`): там сказано, что именно не так, например какой город не найден. Остальная формулировка — на ваше усмотрение.
2. Включите обработку у всех инструментов из `ALL_TOOLS`.
3. Запустите живую проверку ниже и опишите в ячейке после неё, как агент повёл себя с ошибкой.


In [20]:
def format_tool_error(error: ToolException) -> str:
    """Текст ошибки инструмента, который увидит модель."""
    return (
        f"Ошибка инструмента: {error}. "
        "Исправь аргументы (проверь название города или валюты) либо сообщи "
        "пользователю, что таких данных в базе нет. Не выдумывай ответ."
    )

# включаем обработку ToolException у всех инструментов
for current_tool in ALL_TOOLS:
    current_tool.handle_tool_error = format_tool_error



In [21]:
agent = build_agent(llm, ALL_TOOLS)
answer, error_history = chat(agent, [], "Сколько стоит отель в Атлантиде и как туда добраться из Москвы?")
print(answer)
print()
for message in error_history:
    if isinstance(message, ToolMessage):
        print(f"[{message.status}] {message.name}: {message.content[:200]}")


В моей базе нет города «Атлантида». Скорее всего, вы имеете в виду **Атлантис на Пальме в Дубае** (отель Atlantis The Palm) или другое место с таким названием.

Уточните, пожалуйста:
1. Это Дубай (ОАЭ) или другой город/страна?
2. Вас интересует именно отель Atlantis The Palm или просто город в целом?

Если это Дубай — я смогу узнать среднюю цену отеля там и найти маршруты из Москвы.

[error] get_city_info: Ошибка инструмента: Неизвестный город: Atlantis. Доступны: Москва, Санкт-Петербург, Казань, Стамбул, Тбилиси, Ереван, Белград, Дубай. Исправь аргументы (проверь название города или валюты) либо сообщи


**Что сделал агент после ошибки?** Без handle_tool_error агент падал на ToolException. После включения текст ошибки со списком доступных городов возвращается модели в ToolMessage. Модель вызвала get_city_info("Atlantis"), получила ошибку и не стала ничего выдумывать: она сказала, что такого города нет в базе, и спросила, не имеется ли в виду отель Atlantis в Дубае.


### 4.2. Лимит шагов

Агент может зациклиться: например, раз за разом вызывать инструмент с одними и теми же аргументами. У графа LangGraph есть `recursion_limit` (передаётся в `config`), а при его превышении он бросает `GraphRecursionError`.

Напишите `safe_invoke(agent, history, message, max_steps=8) -> (answer, history)` — аналог `chat` с ограничением:

* модель вызывается не больше `max_steps` раз — подберите `recursion_limit` (подсказка: одно обращение к модели и одно исполнение инструментов — это два шага графа);
* в обычном случае возвращает ответ агента и обновлённую историю, как `chat`;
* при превышении лимита функция не падает, а возвращает понятный пользователю текст и исходную историю `history`: незавершённый шаг в неё не попадает.


In [22]:
def safe_invoke(agent, history: list, message: str, max_steps: int = 8) -> tuple[str, list]:
    """Как chat, но не больше max_steps обращений к модели."""
    # одно обращение к модели + одно исполнение инструментов ≈ 2 шага графа LangGraph
    recursion_limit = max_steps * 2
    new_messages = list(history) + [HumanMessage(message)]
    try:
        result = agent.invoke(
            {"messages": new_messages},
            config={"recursion_limit": recursion_limit},
        )
    except GraphRecursionError:
        # незавершённый шаг в историю не кладём — возвращаем исходную
        return (
            f"Превышен лимит шагов ({max_steps}). Не удалось завершить ответ.",
            history,
        )

    updated = result["messages"]
    answer = ""
    for msg in reversed(updated):
        if isinstance(msg, AIMessage) and not msg.tool_calls:
            answer = msg.content if isinstance(msg.content, str) else str(msg.content)
            break
    if not answer:
        last = updated[-1]
        answer = getattr(last, "text", None) or (
            last.content if isinstance(getattr(last, "content", None), str) else str(last)
        )
    return answer, updated



### 4.3. Запрос за пределами возможностей

Агент должен честно разделять, что он может, а что нет. Самый показательный случай — смешанный запрос: часть агент выполнить может, часть нет, а для ещё одной части у него есть похожие, но не те данные.

Запрос ниже состоит из трёх частей:

1. найти самый дешёвый способ добраться из Москвы в Тбилиси - это агент умеет;
2. сказать, какая погода будет там завтра - у агента есть только средняя температура по месяцам, а не прогноз;
3. купить билет — этого агент не умеет.

Хороший ответ выполняет первую часть, честно говорит, что прогноза погоды у него нет (но может назвать среднюю температуру за месяц), и не делает вид, что купил билет.

Посмотрите на ответ и вызовы инструментов. Если агент где-то выдал желаемое за действительное — доработайте `SYSTEM_PROMPT` в части 3, перезапустите ячейки и опишите, что изменили. Если агент сразу ответил хорошо — опишите, по каким признакам в ответе и вызовах это видно.


In [23]:
agent = build_agent(llm, ALL_TOOLS)
answer, scope_history = safe_invoke(
    agent,
    [],
    "Найди самый дешёвый способ добраться из Москвы в Тбилиси, скажи, какая там будет погода завтра, "
    "и сразу купи мне билет.",
)
for message in scope_history:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            print(f"Вызов: {call['name']}({call['args']})")
print()
print(answer)


Вызов: search_routes({'origin': 'Москва', 'destination': 'Тбилиси', 'transport': 'any'})

Самый дешёвый вариант — **прямой рейс Москва → Тбилиси** (самолёт, 3 ч, **220 EUR**).

**Погода:** инструмент `get_climate` выдаёт только *среднюю* температуру за месяц, а не прогноз на конкретный день. Если укажешь месяц (1‑12), смогу назвать среднюю температуру для Тбилиси в этот период.

**Покупка билетов:** я не умею бронировать и оплачивать билеты. Могу только помочь с расчётом маршрута, конвертацией валют или составлением маршрута поездки.


**Как агент справился с каждой из трёх частей запроса и что пришлось поменять в промпте?**

1. Самый дешёвый маршрут агент нашёл через search_routes: прямой самолёт Москва → Тбилиси за 220 EUR.
2. Прогноз на завтра он не выдумал. Он объяснил, что get_climate даёт только среднюю температуру за месяц, и предложил назвать месяц.
3. Покупать билет честно отказался.
Промпт менять не пришлось: в SYSTEM_PROMPT уже были правила «среднее за месяц — не прогноз» и «не бронировать и не покупать».


## 5. Прогон на тестовых вопросах

Прогоним агента на фиксированном наборе вопросов и сохраним трейсы — последовательность шагов агента. По трейсам видно, правильно ли агент выбирал инструменты, а не только каков финальный ответ.

### 5. Трейс агента

**Зачем.** После вызова агента вся история разговора лежит в `result["messages"]` — это та же история, которую возвращают `chat` и `safe_invoke`. Это список объектов LangChain: `HumanMessage` (вопрос пользователя), `AIMessage` (ответ модели: либо запрос на вызов инструментов, либо финальный текст), `ToolMessage` (результат инструмента). Читать такой список неудобно, а сохранить его в JSON напрямую нельзя.

Нужна функция, которая превратит историю в трейс — простой список шагов из обычных словарей. По трейсу сразу видно, какие инструменты агент вызывал, с какими аргументами и что получил в ответ. По трейсам в конце части вы разберёте, как агент справился с тестовыми вопросами.

**Что написать.** Функцию `extract_trace(messages) -> list[dict]`:

* вход — список сообщений из истории агента;
* выход — список словарей, по одному на шаг, в том же порядке, что и сообщения.

Каждое сообщение превращается в один или несколько шагов по правилам:

| Сообщение | Шаг(и) в трейсе |
|---|---|
| `SystemMessage` | пропускается |
| `HumanMessage` | `{"type": "user", "content": <текст>}` |
| `AIMessage` **с** `tool_calls` | если у сообщения есть текст — сначала `{"type": "thought", "content": <текст>}`; затем по шагу на каждый вызов: `{"type": "tool_call", "name": <имя>, "args": <аргументы>}` |
| `ToolMessage` | `{"type": "tool_result", "name": <имя инструмента>, "content": <текст>, "is_error": <True, если status == "error">}` |
| `AIMessage` **без** `tool_calls` | `{"type": "answer", "content": <текст>}` |

**Пример.** Пользователь спросил про дорогу в Белград, модель одним ответом запросила два поиска, второй завершился ошибкой, затем модель ответила.

История (вход):

```python
[
    HumanMessage("Сколько стоит дорога из Москвы в Белград?"),
    AIMessage("", tool_calls=[
        {"name": "search_routes", "args": {"origin": "Москва", "destination": "Стамбул"}, "id": "a"},
        {"name": "search_routes", "args": {"origin": "Стамбул", "destination": "Белград"}, "id": "b"},
    ]),
    ToolMessage("plane 250 EUR", tool_call_id="a", name="search_routes"),
    ToolMessage("Ошибка: сервис недоступен", tool_call_id="b", name="search_routes", status="error"),
    AIMessage("Через Стамбул, от 310 EUR."),
]
```

Трейс (выход):

```python
[
    {"type": "user", "content": "Сколько стоит дорога из Москвы в Белград?"},
    {"type": "tool_call", "name": "search_routes", "args": {"origin": "Москва", "destination": "Стамбул"}},
    {"type": "tool_call", "name": "search_routes", "args": {"origin": "Стамбул", "destination": "Белград"}},
    {"type": "tool_result", "name": "search_routes", "content": "plane 250 EUR", "is_error": False},
    {"type": "tool_result", "name": "search_routes", "content": "Ошибка: сервис недоступен", "is_error": True},
    {"type": "answer", "content": "Через Стамбул, от 310 EUR."},
]
```

Здесь у первого `AIMessage` текст пустой, поэтому шага `thought` нет. Ниже — тот же пример в виде трейса, на котором можно проверить `extract_trace`.

<details>
<summary><strong>Подсказка</strong></summary>

Тип сообщения проверяйте через `isinstance(message, HumanMessage)` и т. д. Текст любого сообщения — `message.text`, вызовы — `message.tool_calls` (список словарей с ключами `name`, `args`, `id`), статус результата — `message.status`.

</details>


In [24]:
def extract_trace(messages: list) -> list[dict]:
    """Превращает историю сообщений агента в список шагов."""
    trace = []
    for message in messages:
        if isinstance(message, SystemMessage):
            continue
        if isinstance(message, HumanMessage):
            trace.append({"type": "user", "content": message.text})
            continue
        if isinstance(message, ToolMessage):
            content = message.content if isinstance(message.content, str) else str(message.content)
            trace.append({
                "type": "tool_result",
                "name": message.name,
                "content": content,
                "is_error": getattr(message, "status", None) == "error",
            })
            continue
        if isinstance(message, AIMessage):
            if message.tool_calls:
                text = message.text if hasattr(message, "text") else (message.content or "")
                if isinstance(text, str) and text.strip():
                    trace.append({"type": "thought", "content": text})
                for call in message.tool_calls:
                    trace.append({
                        "type": "tool_call",
                        "name": call["name"],
                        "args": call["args"],
                    })
            else:
                text = message.text if hasattr(message, "text") else (message.content or "")
                trace.append({"type": "answer", "content": text})
            continue
    return trace



Тестовые вопросы подобраны так, чтобы проверить разные ситуации. **Не меняйте их.** Каждый вопрос задаётся с пустой историей, а маршрут поездки очищается.


In [25]:
TEST_QUESTIONS = [
    # один инструмент
    "Какая погода в Тбилиси в мае?",
    # два инструмента подряд: валюта города → пересчёт
    "Какая валюта в Ереване и сколько в ней будет 100 евро?",
    # нет прямого маршрута: нужна пересадка
    "Как добраться из Казани в Белград? Нужен самый дешёвый вариант.",
    # состояние: маршрут из нескольких городов
    "Составь маршрут: 3 ночи в Стамбуле и 2 в Белграде. Сколько выйдут отели в евро?",
    # фильтры и честный ответ «не найдено»
    "Сколько стоит поезд из Москвы в Дубай?",
    # ошибка инструмента
    "Расскажи, что посмотреть в Атлантиде.",
    # инструмент не нужен
    "Привет! Что ты умеешь?",
    # вне компетенции
    "Забронируй мне отель в Дубае на завтра.",
]

def run_agent(question: str, max_steps: int = 10) -> dict:
    """Один вопрос с чистого листа: ответ и трейс."""
    itinerary_tool.invoke({"action": "clear"})
    agent = build_agent(llm, ALL_TOOLS)
    answer, messages = safe_invoke(agent, [], question, max_steps=max_steps)
    return {"question": question, "answer": answer, "trace": extract_trace(messages)}

def format_step(step: dict) -> str:
    """Одна строка трейса для печати."""
    kind = step.get("type")
    if kind == "user":
        return f"Пользователь: {step['content']}"
    if kind == "thought":
        return f"Мысль: {step['content'][:200]}"
    if kind == "tool_call":
        args = ", ".join(f"{key}={value!r}" for key, value in step["args"].items())
        return f"Вызов: {step['name']}({args})"
    if kind == "tool_result":
        mark = "Ошибка:" if step.get("is_error") else "Результат:"
        return f"   {mark} {step['content'][:150].replace(chr(10), ' / ')}"
    if kind == "answer":
        return f"Ответ: {step['content'][:400]}"
    return str(step)

results = []
for number, question in enumerate(TEST_QUESTIONS, start=1):
    run = run_agent(question)
    results.append(run)
    print(f"=== Вопрос #{number} ===")
    if not run["trace"]:
        print(f"Пользователь: {question}")
    for step in run["trace"]:
        print(format_step(step))
    if not any(step["type"] == "answer" for step in run["trace"]):
        print(f"Ответ: {run['answer']}")
    print()


=== Вопрос #1 ===
Пользователь: Какая погода в Тбилиси в мае?
Вызов: get_climate(city='Тбилиси', month=5)
   Результат: Средняя температура в городе Тбилиси в мае: +17 °C
Ответ: В Тбилиси в мае средняя температура составляет **+17 °C**.

=== Вопрос #2 ===
Пользователь: Какая валюта в Ереване и сколько в ней будет 100 евро?
Вызов: get_city_info(city='Ереван')
   Результат: Ереван, Армения. Валюта: AMD. Часовой пояс: UTC+4. / Отель: около 25000 AMD за ночь. / Розовый город с видом на Арарат. / Достопримечательности: Каскад (бес
Вызов: convert_currency(to_currency='AMD', from_currency='EUR', amount=100)
   Результат: 100.00 EUR = 42000.00 AMD
Ответ: В Ереване (Армения) используется **армянский драм (AMD)**.

По учебному курсу датасета: **100 EUR = 42 000 AMD**.

=== Вопрос #3 ===
Пользователь: Как добраться из Казани в Белград? Нужен самый дешёвый вариант.
Вызов: search_routes(origin='Казань', destination='Белград', transport='any')
   Результат: Прямых маршрутов из Казань в Белград не на

Сводка по прогону. Для каждого вопроса: сколько всего было вызовов инструментов, сколько из них повторов (вызов с теми же аргументами, что уже был), сколько закончились ошибкой. Под строкой — уникальные вызовы каждого инструмента, коротко: только значения аргументов.


In [26]:
from collections import Counter

def short_args(args: dict) -> str:
    """Значения аргументов через «/»: {"origin": "Казань", "destination": "Стамбул"} → «Казань/Стамбул»."""
    return "/".join(str(value) for value in args.values()) or "—"

print(f"{'№':<4}{'вызовов':<9}{'повторов':<10}{'ошибок':<8}инструменты")
for number, run in enumerate(results, start=1):
    calls = [step for step in run["trace"] if step["type"] == "tool_call"]
    keys = [(call["name"], json.dumps(call["args"], ensure_ascii=False, sort_keys=True)) for call in calls]
    repeats = len(keys) - len(set(keys))
    errors = sum(1 for step in run["trace"] if step["type"] == "tool_result" and step["is_error"])
    grouped = ", ".join(
        f"{name} ×{count}" if count > 1 else name for name, count in Counter(call["name"] for call in calls).items()
    )
    print(f"{number:<4}{len(calls):<9}{repeats:<10}{errors:<8}{grouped or '—'}")

    unique: dict[str, list[str]] = {}
    for call in calls:
        values = unique.setdefault(call["name"], [])
        if short_args(call["args"]) not in values:
            values.append(short_args(call["args"]))
    for name, values in unique.items():
        print(f"      {name}: {'; '.join(values)}")


№   вызовов  повторов  ошибок  инструменты
1   1        0         0       get_climate
      get_climate: Тбилиси/5
2   2        0         0       get_city_info, convert_currency
      get_city_info: Ереван
      convert_currency: AMD/EUR/100
3   3        0         0       search_routes ×3
      search_routes: Казань/Белград/any; Стамбул/Казань/any; Белград/Стамбул/any
4   3        0         0       manage_itinerary ×3
      manage_itinerary: add/3/Стамбул; 2/add/Белград; show
5   1        0         0       search_routes
      search_routes: Москва/Дубай/train
6   1        0         1       get_city_info
      get_city_info: Atlantis
7   0        0         0       —
8   0        0         0       —


### Выводы по прогону

Посмотрите трейсы и сводку выше и ответьте, ссылаясь на номера вопросов:

1. Какие вопросы агент решил правильно, а где ошибся? Ошибка была в выборе инструмента, в аргументах или в финальном ответе?
2. Справился ли агент с пересадкой (#3)? Если нет — что бы вы поменяли: описание `search_routes`, системный промпт, сам инструмент?
3. Были ли лишние или повторные вызовы инструментов (колонка «повторов» и число вызовов в сводке)? С чем это связано: с промптом, с описаниями инструментов или с тем, что агенту не хватает какого-то инструмента?
4. Что вы меняли в описаниях инструментов или промпте по ходу работы и как это повлияло на поведение?

**Analysis / Разбор:**

1. Вопросы #1–#4 и #6–#8 решены правильно. В #5 агент правильно сказал, что поезда Москва → Дубай нет, но добавил от себя факты, которых нет в инструментах (про железную дорогу и Каспийское море). Это ошибка в финальном ответе, а не в выборе инструмента.
2. С пересадкой в #3 агент справился. Прямого маршрута не было, поэтому он отдельно нашёл Казань → Стамбул и Стамбул → Белград, итого 290 EUR. Помогли подсказка в описании search_routes и правило о пересадках в промпте.
3. Повторных вызовов не было. Была одна ошибка в #6, она ожидаемая, потому что города Атлантида нет в базе.
4. По ходу работы я указал ISO-коды валют в описании convert_currency, добавил подсказку о пересадках в search_routes, добавил в промпт запрет на покупки и правило «среднее ≠ прогноз», а также включил handle_tool_error. После этого агент перестал падать на ошибках и честно отказывается выполнять то, чего не умеет.

_
